# Microsoft Agent Framework — Azure OpenAI (Válaszok API)

Ebben a kódmintában a **Microsoft Agent Framework (MAF)** használatával hozhat létre egy egyszerű, **Azure OpenAI** által támogatott ügynököt a **Válaszok API** segítségével.

> **Átállási megjegyzés:** Ez a minta korábban a Semantic Kernel-t használta GitHub Modellekkel. Átállt a Microsoft Agent Frameworkre, és a GitHub Modelleket (elavult, 2026 júliusában megszűnik) az Azure OpenAI váltotta fel, amely támogatja a Válaszok API-t. A MAF-ban az `OpenAIChatClient` az Azure OpenAI stabil `/openai/v1/` végpontját célozza meg, és alapértelmezés szerint a Válaszok API-t használja.

Ennek a mintának a célja bemutatni azokat a lépéseket, amelyeket később további kódmintákban alkalmaznak különféle ügynöki mintázatok megvalósításakor.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## A szükséges Python csomagok importálása


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Egy eszköz definiálása

A Microsoft Agent Framework-ben egy **eszköz** egy egyszerű Python függvény, amelyet `@tool` dekorátorral látnak el, és amelyet az ügynök meghívhat. Az alábbiakban definiálunk egy eszközt, amely egy véletlenszerű nyaralási célpontot ad vissza, és elkerüli az előző ismétlését.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Az ügynök létrehozása

Itt létrehozzuk a `TravelAgent` nevű ügynököt.

Ebben a példában nagyon egyszerű utasításokat használunk. Nyugodtan módosítsa ezeket az utasításokat, hogy megfigyelhesse, hogyan változik az ügynök viselkedése.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Az ügynök futtatása

Most már futtathatjuk az ügynököt. Létrehozunk egy `AgentSession`-t, hogy az ügynök emlékezzen a beszélgetésre a fordulók között, majd két `user_inputs`-ot küldünk. Az első egy utazást kér; a második azt mondja, hogy a felhasználónak nem tetszett a javaslat, és egy másikat kér — az ügynök a munkamenet előzményeit és a `get_random_destination` eszközt használja a válaszadáshoz.

Ezeket az üzeneteket módosíthatod, hogy lásd, hogyan reagál az ügynök másként. A válaszok **folyamatosan**, tokenenként érkeznek.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
